# Rotary Position Embedding (RoPE) — a minimal PyTorch implementation

> **RoFormer: Enhanced Transformer with Rotary Position Embedding**
> Su, Lu, Pan, Murtadha, Wen, Liu (2021) — [arXiv:2104.09864](https://arxiv.org/abs/2104.09864)

RoPE injects position into a transformer by **rotating** the query and key vectors by an
angle proportional to their position, instead of **adding** a position vector to the
embedding. Because a dot product of two rotated vectors subtracts their angles, the
attention score ends up depending only on the *relative* distance `n - m`.

It is the position encoding used by LLaMA, Mistral, Qwen, DeepSeek, GPT-NeoX, PaLM and
Gemma.

This notebook is a from-scratch implementation with nothing imported from the rest of the
repo — every function is defined here.

**Contents**

1. Why a position encoding is needed
2. The idea, in two dimensions
3. Implementation
4. Correctness checks
5. Dropping it into an attention layer
6. A transformer block
7. Incremental decoding (KV cache)
8. Practical notes

In [2]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device {DEVICE}")

torch 2.13.0+cu130 | device cuda


## 1. Why a position encoding is needed

Self-attention compares tokens through dot products, and a dot product has no notion of
order. Permuting the input tokens permutes the output rows and changes nothing else.

In [3]:
d = 16
g = torch.Generator().manual_seed(0)
Wq, Wk, Wv = (torch.randn(d, d, generator=g) / math.sqrt(d) for _ in range(3))
x = torch.randn(5, d, generator=g)

def attention(x):
    q, k, v = x @ Wq.T, x @ Wk.T, x @ Wv.T
    return F.softmax(q @ k.T / math.sqrt(d), dim=-1) @ v

perm = torch.randperm(5, generator=g)
print("permutation:", perm.tolist())
print("attend-then-permute == permute-then-attend:",
      torch.allclose(attention(x)[perm], attention(x[perm]), atol=1e-6))

permutation: [1, 4, 2, 0, 3]
attend-then-permute == permute-then-attend: True


## 2. The idea, in two dimensions

Treat a 2-D vector as a complex number and multiply it by a unit phase that depends on the
position. For a query at position `m` and a key at position `n`:

$$q_m = q\,e^{im\theta}, \qquad k_n = k\,e^{in\theta}$$

Their inner product is $\operatorname{Re}[q_m \overline{k_n}]$, and the two phases subtract:

$$\langle q_m, k_n\rangle = \operatorname{Re}\!\big[q\bar k\, e^{i(m-n)\theta}\big]$$

Absolute positions go in; only `m - n` survives. In real coordinates the multiplication is
a rotation matrix:

$$R(\theta) = \begin{pmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{pmatrix}$$

Two properties are what make this work: rotations **compose** ($R_aR_b = R_{a+b}$, so the
angles can cancel) and they are **orthogonal** ($\|Rx\| = \|x\|$, so attention scores are
not rescaled).

In [4]:
def rotate_2d(v, angle):
    c, s = math.cos(angle), math.sin(angle)
    return torch.tensor([v[0] * c - v[1] * s, v[0] * s + v[1] * c])

theta = math.radians(45)
q = torch.tensor([1.0, 0.0])
k = torch.tensor([0.8, 0.6])

print(" m    n   n-m    <R_m q, R_n k>")
for m, n in [(2, 7), (10, 15), (100, 105), (2, 4), (10, 12)]:
    score = (rotate_2d(q, m * theta) * rotate_2d(k, n * theta)).sum()
    print(f"{m:>3}  {n:>3}  {n - m:>4}       {score:+.6f}")

print("\nPairs sharing a gap share a score, wherever they sit in the sequence.")
print(f"norm before rotation {q.norm():.4f}, after {rotate_2d(q, 3 * theta).norm():.4f}")

 m    n   n-m    <R_m q, R_n k>
  2    7     5       -0.141421
 10   15     5       -0.141421
100  105     5       -0.141421
  2    4     2       -0.600000
 10   12     2       -0.600000

Pairs sharing a gap share a score, wherever they sit in the sequence.
norm before rotation 1.0000, after 1.0000


## 3. Implementation

A head of dimension `d` is split into `d/2` two-dimensional planes, and plane `i` rotates
at its own rate:

$$\theta_i = \text{base}^{-2i/d}, \qquad i = 0 \dots d/2 - 1$$

Low planes turn quickly (a full turn every few tokens) and resolve nearby positions; high
planes turn slowly and carry long-range position. `base` defaults to 10000.

Stacked up, this is a block-diagonal rotation matrix, but it is never built — it has only
`2d` non-zero entries out of `d²`. The equivalent elementwise form is:

$$x_{\text{rope}} = x \odot \cos(m\Theta) \;+\; \texttt{rotate\_half}(x) \odot \sin(m\Theta)$$

where `rotate_half` maps `(x₁, x₂) → (−x₂, x₁)` within each plane. Expanding the first
component gives $x_1\cos - x_2\sin$ and the second $x_2\cos + x_1\sin$ — exactly the
rotation.

The angles depend only on position, so `cos` and `sin` are precomputed once into a table.

In [5]:
def rope_frequencies(head_dim: int, base: float = 10000.0) -> torch.Tensor:
    """theta_i = base ** (-2i/d) for i in [0, d/2)."""
    assert head_dim % 2 == 0, "head_dim must be even"
    i = torch.arange(0, head_dim, 2, dtype=torch.float32)
    return base ** (-i / head_dim)


def build_rope_cache(head_dim: int, max_seq_len: int, base: float = 10000.0,
                     device=None, dtype=torch.float32):
    """Precompute cos/sin tables of shape (max_seq_len, head_dim)."""
    theta = rope_frequencies(head_dim, base).to(device)
    pos = torch.arange(max_seq_len, dtype=torch.float32, device=device)
    angles = torch.outer(pos, theta)                 # (T, d/2)
    angles = torch.cat([angles, angles], dim=-1)     # (T, d), duplicated per half
    return angles.cos().to(dtype), angles.sin().to(dtype)

In [7]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """(x1, x2) -> (-x2, x1), pairing each channel with the one d/2 away."""
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
               offset: int = 0) -> torch.Tensor:
    """
    Rotate x by its position.

    x            (..., seq_len, head_dim)
    cos, sin     (max_seq_len, head_dim), from build_rope_cache
    offset       absolute position of x[..., 0, :] - non-zero when decoding with a cache
    """
    seq_len = x.shape[-2]
    c = cos[offset:offset + seq_len].to(x.dtype)
    s = sin[offset:offset + seq_len].to(x.dtype)
    return x * c + rotate_half(x) * s

That is the whole method: two tables, one `chunk`/`cat`, and one multiply-add. It costs
`O(d)` per token and adds **no learned parameters**.

A note on channel pairing. The paper pairs adjacent channels — `(x₁,x₂), (x₃,x₄), …` —
while LLaMA and Hugging Face pair across the halves, `(x₁, x_{1+d/2}), …`, which is why
`rotate_half` is a `chunk` plus a `cat` rather than a strided reshape. The two differ by a
fixed permutation of channels that `W_q` and `W_k` absorb during training, so either is
fine as long as queries and keys use the same one.

## 4. Correctness checks

The property that matters: shifting a block of tokens to a different absolute position
must leave every attention score unchanged.

In [8]:
head_dim, seq_len = 64, 16
cos, sin = build_rope_cache(head_dim, max_seq_len=4096)

g = torch.Generator().manual_seed(1)
q = torch.randn(2, 4, seq_len, head_dim, generator=g)   # (batch, heads, T, head_dim)
k = torch.randn(2, 4, seq_len, head_dim, generator=g)

def scores(offset):
    return apply_rope(q, cos, sin, offset) @ apply_rope(k, cos, sin, offset).transpose(-1, -2)

base = scores(0)
for off in [1, 8, 64, 512, 2048]:
    drift = (scores(off) - base).abs().max().item()
    print(f"  offset {off:>5}: max score change {drift:.2e}")

print(f"\nnorm preserved      : "
      f"{(apply_rope(q, cos, sin).norm(dim=-1) - q.norm(dim=-1)).abs().max().item():.2e}")
print(f"position 0 unchanged: "
      f"{torch.allclose(apply_rope(q[..., :1, :], cos, sin), q[..., :1, :], atol=1e-6)}")

  offset     1: max score change 9.54e-06
  offset     8: max score change 1.14e-05
  offset    64: max score change 2.86e-05
  offset   512: max score change 1.60e-04
  offset  2048: max score change 7.48e-04

norm preserved      : 9.54e-07
position 0 unchanged: True


The residual is floating-point error, not a property of the method: `m * theta` grows with
position, so `cos`/`sin` lose relative precision as the offset increases. In exact
arithmetic the change is zero. This is also why the tables are built in `float32` even when
the model runs in `bfloat16`.

For comparison, the same test with an additive absolute position encoding, which is what
RoPE replaces:

In [9]:
def sinusoidal_pe(seq_len, dim, device=None):
    pos = torch.arange(seq_len, dtype=torch.float32, device=device).unsqueeze(1)
    i = torch.arange(0, dim, 2, dtype=torch.float32, device=device)
    div = torch.exp(-math.log(10000.0) * i / dim)
    pe = torch.zeros(seq_len, dim, device=device)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
    return pe

pe = sinusoidal_pe(4096, head_dim)

def scores_additive(offset):
    qa = q + pe[offset:offset + seq_len]
    ka = k + pe[offset:offset + seq_len]
    return qa @ ka.transpose(-1, -2)

base_add = scores_additive(0)
for off in [1, 8, 64, 512, 2048]:
    print(f"  offset {off:>5}: max score change {(scores_additive(off) - base_add).abs().max().item():8.3f}")

  offset     1: max score change    7.368
  offset     8: max score change   26.518
  offset    64: max score change   30.771
  offset   512: max score change   39.909
  offset  2048: max score change   45.823


## 5. Dropping it into an attention layer

RoPE is applied to `q` and `k` after the projection and before the dot product. `v` is
left alone, and nothing is added to the residual stream.

In [10]:
class RoPESelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int = 4096,
                 rope_base: float = 10000.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

        cos, sin = build_rope_cache(self.head_dim, max_seq_len, rope_base)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def forward(self, x, offset: int = 0, is_causal: bool = True):
        b, t, _ = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        shape = (b, t, self.n_heads, self.head_dim)
        q, k, v = (z.view(shape).transpose(1, 2) for z in (q, k, v))   # (B, H, T, Dh)

        q = apply_rope(q, self.cos, self.sin, offset)                  # position enters
        k = apply_rope(k, self.cos, self.sin, offset)                  # here, on q and k

        y = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
        return self.out(y.transpose(1, 2).reshape(b, t, -1))

In [11]:
attn = RoPESelfAttention(d_model=256, n_heads=8).eval()
x = torch.randn(2, 32, 256)

with torch.no_grad():
    y = attn(x)
print(f"input {tuple(x.shape)} -> output {tuple(y.shape)}")

# the same tokens processed at a later absolute offset give the same result
with torch.no_grad():
    print("output at offset 0 == output at offset 1000:",
          torch.allclose(attn(x, offset=0), attn(x, offset=1000), atol=1e-5))

input (2, 32, 256) -> output (2, 32, 256)
output at offset 0 == output at offset 1000: True


## 6. A transformer block

Nothing else in the model changes. There is no position embedding table and no positional
term added to the token embeddings.

In [12]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len=4096):
        super().__init__()
        self.n1 = nn.LayerNorm(d_model)
        self.attn = RoPESelfAttention(d_model, n_heads, max_seq_len)
        self.n2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                 nn.Linear(4 * d_model, d_model))

    def forward(self, x, offset: int = 0):
        x = x + self.attn(self.n1(x), offset)
        return x + self.mlp(self.n2(x))


class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=4, max_seq_len=4096):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList(Block(d_model, n_heads, max_seq_len)
                                    for _ in range(n_layers))
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, offset: int = 0):
        x = self.tok(idx)                      # no position embedding added here
        for blk in self.blocks:
            x = blk(x, offset)
        return self.head(self.norm(x))


model = TinyLM(vocab_size=1000).to(DEVICE)
idx = torch.randint(0, 1000, (2, 64), device=DEVICE)
print(f"logits: {tuple(model(idx).shape)}")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"of which positional: {sum(p.numel() for n, p in model.named_parameters() if 'pos' in n)}")

logits: (2, 64, 1000)
parameters: 3,667,456
of which positional: 0


## 7. Incremental decoding (KV cache)

During generation each new token is processed on its own, so it must be rotated by its
**absolute** position rather than by 0. That is what the `offset` argument is for; passing
the wrong value silently rotates the cache into nonsense.

In [13]:
head_dim = 32
cos, sin = build_rope_cache(head_dim, 512)
g = torch.Generator().manual_seed(2)
tokens = torch.randn(1, 2, 10, head_dim, generator=g)      # (B, H, T, Dh)

full = apply_rope(tokens, cos, sin)                        # whole sequence at once

incremental = torch.cat([                                  # one token at a time
    apply_rope(tokens[:, :, t:t + 1, :], cos, sin, offset=t) for t in range(10)
], dim=-2)

wrong = torch.cat([                                        # forgetting the offset
    apply_rope(tokens[:, :, t:t + 1, :], cos, sin, offset=0) for t in range(10)
], dim=-2)

print(f"correct offsets : max difference {(full - incremental).abs().max().item():.2e}")
print(f"offset always 0 : max difference {(full - wrong).abs().max().item():.4f}")

correct offsets : max difference 0.00e+00
offset always 0 : max difference 4.8093


## 8. Practical notes

- **Applied to `q` and `k` only**, never to `v`, and never added to the residual stream.
- **Applied in every attention layer**, not once at the input like an absolute encoding.
- **No learned parameters.** The cos/sin tables are buffers, not parameters, so they are
  usually registered with `persistent=False` and rebuilt on load.
- **Build the tables in `float32`** and cast to the activation dtype at use. Computing the
  angles directly in `bfloat16` loses precision at large positions.
- **`base` controls the useful range.** 10000 was chosen for contexts of ~512 tokens;
  long-context models raise it (LLaMA-3 uses 500000) and/or interpolate positions.
- **`max_seq_len` is only a table size.** RoPE has no hard length ceiling the way a learned
  embedding table does, but a model still degrades on relative distances it never saw in
  training — which is what position interpolation, NTK-aware scaling and YaRN address.
- **Queries and keys must use the same channel pairing** (`rotate_half` vs. interleaved) and
  the same `base`.

## Reference

Su et al., 2021 — *RoFormer: Enhanced Transformer with Rotary Position Embedding*,
[arXiv:2104.09864](https://arxiv.org/abs/2104.09864)